# rotation-matrix-3d-y-axis — ex8: inverse rotation = transpose — numerical sweep

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rotation-matrix-3d-y-axis`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** Right-hand rotation by `θ` about Y:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```
Anything along Y stays put (middle row `[0,1,0]`); the X-Z plane rotates.

**Acting on data.** Column-vector form: `v' = R @ v`. Batch of row-vectors `(N, 3)`: `points' = points @ R.T`. Composition: `R(α) @ R(β) = R(α + β)` for single-axis rotations; multi-axis rotations don't commute.

**Numerical truth.** Rotation matrices are orthogonal: `R @ R.T = I` and `R.inverse() == R.T`. Floating-point composition accumulates ~1e-7 error per matmul.

### Exercise 8 — inverse rotation = transpose — numerical sweep

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Verify the orthogonality identity `R(θ)⁻¹ = R(-θ) = R(θ).T` numerically across an angle sweep and quantify floating-point deviation.
> Keywords: orthogonality, inverse, transpose, numerical-error, sweep
> ```

**KCs targeted:** `rotation-matrix-y-construct`, `rotation-orthogonality`

Implement `ex8_inverse_error_sweep(angles)`. Given a 1-D tensor of `M` angles, return a `dict` with three `(M,)` tensors:

```
{
  'err_inv_vs_negtheta': |R(θ)⁻¹  -  R(-θ)|.max() per angle,
  'err_inv_vs_transpose': |R(θ)⁻¹  -  R(θ).T|.max() per angle,
  'err_RRT_minus_I':       |R(θ) @ R(θ).T  -  I|.max() per angle,
}
```

Build `R(θ)`, compute its inverse with `t.linalg.inv`, then compare against (a) `R(-θ)` built from scratch, (b) `R(θ).T`, and (c) the identity-preservation check `R @ R.T == I`. Use `.max()` to collapse the 3×3 absolute-difference matrix to a scalar per angle.

The test verifies all three error tensors are at floating-point noise level (< 1e-5) and prints a comparison plot of error magnitude vs angle.

In [ ]:
def ex8_inverse_error_sweep(angles: Tensor) -> dict:
    """Return a dict of (M,) per-angle max-abs error tensors."""
    raise NotImplementedError()


def _test_ex8():
    import math

    angles = t.linspace(-math.pi, math.pi, 21)
    out = ex8_inverse_error_sweep(angles)

    # Structure.
    expected_keys = {'err_inv_vs_negtheta', 'err_inv_vs_transpose', 'err_RRT_minus_I'}
    assert set(out.keys()) == expected_keys, f'expected {expected_keys}, got {set(out.keys())}'

    for k in expected_keys:
        assert isinstance(out[k], Tensor), f'{k} must be a Tensor, got {type(out[k])}'
        assert out[k].shape == (21,), f'{k} should be (21,), got {tuple(out[k].shape)}'

    # All errors should be at floating-point noise level for a well-conditioned R.
    for k in expected_keys:
        max_err = out[k].max().item()
        assert max_err < 1e-4, f'{k}: max error {max_err} too large — orthogonality should hold'

    # Sanity: transpose-vs-inv should be even smaller than inv-vs-negtheta, since
    # transpose is exact and -theta involves a fresh cos/sin pair.
    median_transpose = out['err_inv_vs_transpose'].median().item()
    assert median_transpose < 1e-5, f'transpose ought to be near-exact: got {median_transpose}'

    # Visualize error magnitude vs angle.
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogy(angles.numpy(), out['err_inv_vs_negtheta'].numpy(),
                'o-', label='|R⁻¹ - R(-θ)|', alpha=0.8)
    ax.semilogy(angles.numpy(), out['err_inv_vs_transpose'].numpy(),
                's-', label='|R⁻¹ - R.T|', alpha=0.8)
    ax.semilogy(angles.numpy(), out['err_RRT_minus_I'].numpy(),
                '^-', label='|R @ R.T - I|', alpha=0.8)
    ax.set_xlabel('θ (rad)'); ax.set_ylabel('max abs error (log scale)')
    ax.set_title('Orthogonality identities hold at float32 precision')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    print(f'max error across all checks: {max(v.max().item() for v in out.values()):.2e}')
    print(f'median |R⁻¹ - R.T| = {median_transpose:.2e} (effectively zero — they ARE equal)')
    _dd_passed.add('ex8')
    print("ex8 ✓")

_test_ex8()

<details><summary>Solution</summary>

```python
def ex8_inverse_error_sweep(angles: Tensor) -> dict:
    err_inv_neg, err_inv_T, err_RRT = [], [], []
    I = t.eye(3)
    for theta in angles:
        c, s = t.cos(theta).item(), t.sin(theta).item()
        R = t.tensor([
            [c,   0.0, s  ],
            [0.0, 1.0, 0.0],
            [-s,  0.0, c  ],
        ])
        cn, sn = t.cos(-theta).item(), t.sin(-theta).item()
        R_neg = t.tensor([
            [cn,   0.0, sn ],
            [0.0,  1.0, 0.0],
            [-sn,  0.0, cn ],
        ])
        R_inv = t.linalg.inv(R)
        err_inv_neg.append((R_inv - R_neg).abs().max())
        err_inv_T.append((R_inv - R.T).abs().max())
        err_RRT.append((R @ R.T - I).abs().max())
    return {
        'err_inv_vs_negtheta': t.stack(err_inv_neg),
        'err_inv_vs_transpose': t.stack(err_inv_T),
        'err_RRT_minus_I': t.stack(err_RRT),
    }
```

**Why transpose = inverse for rotation matrices.** Rotation matrices are *orthogonal* — their rows (and columns) form an orthonormal basis. The defining property is `R @ R.T = I`, which by definition means `R.T = R⁻¹`. Computing an inverse via `t.linalg.inv` solves a linear system; computing a transpose is free. For rotations, always use `.T`.

**Why the errors aren't exactly zero.** `t.linalg.inv` uses LU decomposition and accumulates roundoff. The transpose route is exact (it's just a stride swap), so `|R⁻¹ - R.T|` ≈ 1e-7 not 0 — that's the inv path's error, not the transpose's.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()